**Group Name:** CS5489 Group ___  
**Members:** ___

# CS5489 — Test Code Template (2026)

**Purpose:** Verify that the submitted `model.pth` can be loaded and used to predict RLE masks from RGB test images. **No ground-truth labels are required.** Students must complete the marked sections so TAs can run the notebook by changing only the paths below.

**This submission:** the model is `ResNet34-UNet + ASPP + SCSE + deep supervision`
(see `aigc_v2/model.py`). Inference is **tiled at native resolution** with flip-TTA,
because the input images are up to 4000x4000 and the AI-generated pixels are
high-frequency. Preprocessing (ImageNet mean/std, no resize) matches training.

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn.functional as F

# ---- TA: only these paths need to be changed ------------------------------
test_dir = Path("F:/AIGC数据集/AIGC Detection Dataset 2026/test/images")
model_path = Path("model.pth")          # trained checkpoint (plain state_dict)
output_csv = Path("submission.csv")
# directory that contains rle.py / model.py / inference.py
code_dir = Path("aigc_v2")
# ---------------------------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if not code_dir.is_absolute():
    code_dir = (Path.cwd() / code_dir).resolve()
sys.path.insert(0, str(code_dir))
from model import build_model as _build_model          # noqa: E402
from inference import predict_prob_map                 # noqa: E402
print("model code:", code_dir)

## Student implementation

**Students:** Complete the next three functions. Include all model architecture definitions and dependencies in the submitted notebook (or accompanying files). Your preprocessing must match training. `predict_binary_mask` must return a 2D binary array of shape `(original_height, original_width)`. Do not use test labels.

In [ ]:
MODEL_META = {}


def build_model():
    """Instantiate the trained segmentation architecture."""
    return _build_model(
        MODEL_META.get("arch", "resnet34"),
        pretrained=False,
        hf=MODEL_META.get("hf", False),
        base=MODEL_META.get("base", 32),
        deep_sup=MODEL_META.get("deep_sup", True),
    )


def load_model(model_path, device):
    """Load model.pth into the architecture and set eval mode."""
    global MODEL_META
    meta_path = Path(model_path).with_name("model_meta.json")
    if meta_path.is_file():
        MODEL_META = json.loads(meta_path.read_text(encoding="utf-8"))
        print("meta:", {k: MODEL_META[k] for k in ("arch", "hf", "threshold")
                        if k in MODEL_META})
    model = build_model().to(device)
    checkpoint = torch.load(model_path, map_location=device, weights_only=True)
    if isinstance(checkpoint, dict) and "model" in checkpoint and "state_dict" not in checkpoint:
        checkpoint = checkpoint["model"]
    model.load_state_dict(checkpoint)
    model.eval()
    return model


def predict_binary_mask(model, image, device):
    """Predict a full-resolution binary mask from a PIL RGB image.

    Tiled native-resolution inference + 4-way flip TTA, then the same
    post-processing that produced our leaderboard submission:

    * threshold tuned on the held-out validation split (stored in
      ``model_meta.json``, 0.70 for our model);
    * if that threshold wipes the image out completely while the maximum
      probability is still >= 0.45, re-threshold that image relatively
      (+0.6 mean IoU on validation: 21/500 images were otherwise blank).
    """
    threshold = MODEL_META.get("threshold", 0.5)
    tile = int(MODEL_META.get("val_tile", 512))
    stride = int(MODEL_META.get("val_stride", 384))
    amp_dtype = torch.bfloat16 if (device.type == "cuda" and
                                   torch.cuda.is_bf16_supported()) else None
    prob = predict_prob_map(model, image, tile=tile, stride=stride,
                            tta_flips=4, scales=(1.0,), device=device,
                            amp_dtype=amp_dtype)
    mask = prob >= threshold
    if not mask.any():
        peak = float(prob.max())
        if peak >= 0.45:
            mask = prob >= max(0.25 * threshold, 0.5 * peak)
    return mask.astype(np.uint8)

## Shared RLE encoder (do not change)

Uses **column-major** flattening, **1-based** starting positions, and `start length` pairs. Empty mask → empty string.

In [ ]:
def mask_to_rle(mask):
    mask = np.asarray(mask)
    if mask.ndim != 2 or not np.isin(mask, [0, 1]).all():
        raise ValueError("Predicted mask must be 2D and contain only 0/1")
    pixels = mask.astype(np.uint8).ravel(order="F")
    padded = np.pad(pixels, (1, 1), constant_values=0)
    changes = np.flatnonzero(padded[1:] != padded[:-1])
    starts = changes[::2] + 1
    lengths = changes[1::2] - changes[::2]
    return " ".join(str(v) for pair in zip(starts, lengths) for v in pair)

assert mask_to_rle(np.array([[0, 0, 1, 1, 1], [0, 0, 0, 0, 1]], dtype=np.uint8)) == "5 1 7 1 9 2"
print("RLE encoder check passed")

## Inference verification and CSV export

This cell must run **without access to any test ground-truth labels**. It checks every predicted mask shape and creates a Kaggle-compatible CSV with `ID,Predicted`. IDs are image filename stems and retain leading zeros.

In [ ]:
def run_test(test_dir, model_path, output_csv, device):
    if not test_dir.is_dir():
        raise FileNotFoundError(f"Test image directory not found: {test_dir}")
    if not model_path.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {model_path}")

    extensions = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
    images = sorted(p for p in test_dir.iterdir()
                    if p.is_file() and p.suffix.lower() in extensions)
    if not images:
        raise ValueError(f"No images found in {test_dir}")
    ids = [p.stem for p in images]
    if len(ids) != len(set(ids)):
        raise ValueError("Duplicate image IDs across file extensions")

    model = load_model(model_path, device)
    rows = []
    for index, path in enumerate(images, 1):
        with Image.open(path) as im:
            rgb = im.convert("RGB")
            height, width = rgb.height, rgb.width
            with torch.inference_mode():
                mask = np.asarray(predict_binary_mask(model, rgb, device))
        if mask.shape != (height, width):
            raise ValueError(f"{path.name}: expected {(height, width)}, got {mask.shape}")
        rle = mask_to_rle(mask)
        rows.append({"ID": path.stem, "Predicted": rle})
        if index == 1 or index % 100 == 0 or index == len(images):
            print(f"Processed {index}/{len(images)} images")

    df = pd.DataFrame(rows, columns=["ID", "Predicted"])
    df.to_csv(output_csv, index=False)
    print(f"PASS: checkpoint loaded and {len(df)} masks predicted")
    print(f"Saved: {output_csv}")
    print(df.assign(Predicted=df.Predicted.str.slice(0, 70)).head())
    return df

submission = run_test(test_dir, model_path, output_csv, device)

## Submission checklist

- [ ] Run all cells successfully with the submitted `model.pth`.
- [ ] Model loads in evaluation mode without training data or hidden labels.
- [ ] Every test image produces a full-resolution 0/1 mask.
- [ ] `submission.csv` contains one row per test image, with columns `ID,Predicted`.
- [ ] Submit the runnable `test.ipynb`, checkpoint, and any required local model code/dependencies as instructed by the course.

**For TAs:** Change `test_dir` and `model_path` in the configuration cell, then Run All. The notebook verifies inference and produces predictions; it does **not** calculate the Kaggle score.

**Runtime:** ~1-3 s/image on a small GPU (tiled 512px inference with 4-way flip TTA), so the 500 public test images take roughly 10-25 minutes.